# Notebook 4: Strategic Financial Simulations & Growth Forecasting (2025–2026)

## 1. Overview

This notebook extends the diagnostic analysis of Notebook 2 into forward-looking financial modeling. Using the Star Schema dimensional data (Notebook 3) as the computational baseline, the notebook addresses two strategic planning requests:

| Module | Scope | Objective |
|--------|-------|-----------|
| 2 | Website Channel 2025 | Re-cost structure modeling; Break-even Point (BEP) calculation |
| 3 | 130% BEP Target Assessment | Corporate revenue impact simulation |
| 4 | Strategy A+ 2026 | Scenario-based net profit forecasting (base case & optimistic case) |

In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# Configure pandas options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Define path to dimensional tables
processed_dir = '../data/processed/'
if not os.path.exists(processed_dir):
    processed_dir = 'data/processed/'

# Ingest Star Schema Tables
try:
    fact_order = pd.read_csv(os.path.join(processed_dir, 'fact_order.csv'))
    dim_customer = pd.read_csv(os.path.join(processed_dir, 'dim_customer.csv'))
    dim_product = pd.read_csv(os.path.join(processed_dir, 'dim_product.csv'))
    dim_date = pd.read_csv(os.path.join(processed_dir, 'dim_date.csv'))
    dim_channel = pd.read_csv(os.path.join(processed_dir, 'dim_channel.csv'))
    print(f"Successfully loaded Star Schema. Transactions loaded: {len(fact_order):,}")
except FileNotFoundError as e:
    print(f"Error loading Star Schema tables: {e}. Ensure Notebook 3 ran successfully.")

## 2. Website Channel Re-Costing & Break-even Point (BEP), 2025

ITB introduces a revised cost structure for the Website channel effective 2025.

### 2.1. Fixed Cost Structure (Annual)

| Cost Item | Amount (VND) |
|-----------|-------------|
| Website Design & Infrastructure | 500,000,000 |
| Customer Service Staff (55 staff) | 13,200,000,000 |
| Office Rent | 3,600,000,000 |
| Storage Fee | 1,200,000,000 |
| CRM System | 200,000,000 |
| Affiliate Marketing | 1,000,000,000 |
| Other Overheads | 1,000,000,000 |
| **Total Annual Fixed Cost** | **20,700,000,000** |

### 2.2. Variable Cost Structure (Per Order)

| Cost Item | Rate |
|-----------|------|
| Shipping | 40,000 VND/order |
| Payment Gateway Fee | 2% of `product_price` |
| Email Marketing Acquisition | 16,667 VND/order (derived from 500 VND/email at 3% conversion) |
| Installation Fee (Product C only) | 10% of `product_price` |

In [ ]:
print("--- 1. Initializing 2025 Website Cost Engine ---")

# Define total annual fixed costs for Website in 2025
annual_fixed_cost_website_2025 = 20_700_000_000 # 20.7 Billion VND

# Extract historical website transactions to model typical pricing behaviors
# Merge fact_order with dimensions to get product profile and channel name
df_joined = pd.merge(fact_order, dim_channel, on='channel_key', how='left')
df_joined = pd.merge(df_joined, dim_product[['product_key', 'product_profile']], on='product_key', how='left')

# Filter for successfully delivered Website transactions
df_website_delivered = df_joined[
    (df_joined['order_status_key'] == 1) & # successfully delivered
    (df_joined['channel_name'] == 'website')
].copy()

# Calculate dynamic variable cost for each transaction based on 2025 rules
def calculate_website_2025_variable_cost(row):
    price = row['product_price']
    product = row['product_profile']
    
    # Standard variables
    shipping_fee = 40000
    email_fee = 16667 # 500 * (100/3)
    gateway_fee = price * 0.02 # 2% of price
    
    # Product C installation fee (10% of price)
    installation_fee = (price * 0.10) if product == 'product_C' else 0
    
    return shipping_fee + email_fee + gateway_fee + installation_fee

# Run simulation of 2025 costs on Website historical orders
df_website_delivered['variable_cost_2025'] = df_website_delivered.apply(calculate_website_2025_variable_cost, axis=1)
df_website_delivered['contribution_margin_2025'] = df_website_delivered['product_price'] - df_website_delivered['variable_cost_2025']

# Compute average order price and average contribution margin
avg_price_per_order = df_website_delivered['product_price'].mean()
avg_cm_per_order = df_website_delivered['contribution_margin_2025'].mean()

print(f"Average Order Price: {avg_price_per_order:,.0f} VND")
print(f"Average Contribution Margin per Order (2025 Rules): {avg_cm_per_order:,.0f} VND")


print("\n--- 2. Calculating 2025 Website Break-even Point (BEP) ---")

# Calculate Break-even Point (BEP) in terms of order volume
bep_orders_yearly = annual_fixed_cost_website_2025 / avg_cm_per_order
bep_orders_monthly = bep_orders_yearly / 12
bep_orders_weekly = bep_orders_yearly / 52.14

# Calculate corresponding revenue needed for breakeven
bep_revenue_yearly = bep_orders_yearly * avg_price_per_order

print(f"Annual Fixed Costs to cover: {annual_fixed_cost_website_2025:,.0f} VND")
print(f"Break-even Volume (Year): {bep_orders_yearly:,.0f} orders/year (Target Revenue: {bep_revenue_yearly:,.0f} VND)")
print(f"Break-even Volume (Month): {bep_orders_monthly:,.0f} orders/month")
print(f"Break-even Volume (Week): {bep_orders_weekly:,.0f} orders/week")

#### Key Findings — Website BEP 2025

| Metric | Value |
|--------|-------|
| Average Order Value (AOV) | 11,278,225 VND |
| Contribution Margin per Order | 10,648,159 VND |
| Contribution Margin Ratio | ~94.4% |
| Break-even Point (BEP) | 1,944 orders/year (~162 orders/month) |
| Break-even Revenue | 21.9 Billion VND |

The Website channel's variable cost structure is highly lean, yielding a 94.4% contribution margin ratio. At this margin, only 1,944 annual orders are required to recover the 20.7 B VND fixed cost base — a threshold achievable at approximately 37 orders per week.

## 3. Assessment: 130% BEP as a Growth Target (2025)

This section evaluates the corporate revenue consequences of setting the Website channel's 2025 volume target at 130% of its BEP, holding all other channels at 2024 actuals.

### 3.1. Simulation Logic

| Parameter | Value |
|-----------|-------|
| 130% BEP target volume | 2,527 orders/year |
| Implied Website revenue (2025) | 28.5 Billion VND |
| Actual Website orders (2024) | 186,342 orders/year |
| Actual Website revenue (2024) | 2.26 Trillion VND |
| Volume reduction implied | -98.6% |

**The BEP Displacement Effect:** While BEP is a valid cost-coverage metric, applying it as a growth ceiling would reduce the highest-margin channel's transaction volume by 98.6% relative to 2024 actuals — resulting in a projected corporate revenue contraction of **-25.61%** (-2.2 Trillion VND).

In [ ]:
print("--- 1. Simulating 130% BEP Target Impact on 2025 Corporate Revenue ---")

# Import plotting libraries locally to prevent NameError
import matplotlib.pyplot as plt
import seaborn as sns

# Define assets directory path resolution locally (to avoid NameError between separate notebook files)
assets_dir = '../assets/charts/'
if not os.path.exists('../assets/'):
    assets_dir = 'assets/charts/'
os.makedirs(assets_dir, exist_ok=True)

# Step 1.1: Map dim_date to get the 'year' column in df_joined
df_joined_with_date = pd.merge(df_joined, dim_date[['date_key', 'year']], on='date_key', how='left')

# Re-isolate Website successful transactions (now with 'year' column included)
df_website_delivered = df_joined_with_date[
    (df_joined_with_date['order_status_key'] == 1) & 
    (df_joined_with_date['channel_name'] == 'website')
].copy()

# A. Calculate actual Website 2024 revenue and orders
actual_web_rev_2024 = df_website_delivered[df_website_delivered['year'] == 2024]['product_price'].sum()
actual_web_orders_2024 = len(df_website_delivered[df_website_delivered['year'] == 2024])

# B. Calculate total 2024 corporate revenue (Only for successful orders in the year 2024)
total_corp_rev_2024 = df_joined_with_date[
    (df_joined_with_date['order_status_key'] == 1) & 
    (df_joined_with_date['year'] == 2024)
]['product_price'].sum()

# C. Calculate revenue of other channels in 2024 (excluding Website)
other_channels_rev_2024 = total_corp_rev_2024 - actual_web_rev_2024

# D. Calculate 130% BEP projected Website revenue for 2025
projected_web_orders_2025 = bep_orders_yearly * 1.30
projected_web_rev_2025 = projected_web_orders_2025 * avg_price_per_order

# E. Calculate projected total corporate revenue for 2025
projected_total_corp_rev_2025 = other_channels_rev_2024 + projected_web_rev_2025

# F. Calculate YoY Growth Rate
overall_yoy_growth_rate = (projected_total_corp_rev_2025 / total_corp_rev_2024 - 1) * 100

print(f"Website Actual 2024 Orders: {actual_web_orders_2024:,} | Revenue: {actual_web_rev_2024:,.0f} VND")
print(f"Website Projected 2025 Orders (130% BEP): {projected_web_orders_2025:,.0f} | Revenue: {projected_web_rev_2025:,.0f} VND")
print(f"Other Channels 2024 Revenue (Static): {other_channels_rev_2024:,.0f} VND")
print(f"Projected 2025 Total Corporate Revenue: {projected_total_corp_rev_2025:,.0f} VND")
print(f"Projected Corporate Revenue YoY Change: {overall_yoy_growth_rate:.2f}%")


print("\n--- 2. Visualizing the Revenue Drop ---")

# Prepare data for plotting
comparison_data = pd.DataFrame({
    'Metric': ['2024 Actual Revenue', '2025 Projected Revenue\n(Website at 130% BEP)'],
    'Revenue (Billion VND)': [total_corp_rev_2024 / 1e9, projected_total_corp_rev_2025 / 1e9]
})

plt.figure(figsize=(10, 7))
ax_bar = sns.barplot(
    data=comparison_data,
    x='Metric',
    y='Revenue (Billion VND)',
    hue='Metric',                # Specified to avoid FutureWarning
    legend=False,                # Specified to avoid FutureWarning
    palette=['#1F77B4', '#D9383A']
)

# Customize plot
plt.title('NEGATIVE IMPACT ON TOTAL COMPANY REVENUE IF ONLY TARGETING 130% OF WEBSITE BEP', fontsize=12, fontweight='bold', pad=20)
plt.ylabel('Total Company Revenue (Billion VND)', fontsize=11)
plt.xlabel('')
plt.ylim(0, max(comparison_data['Revenue (Billion VND)']) * 1.2)
plt.grid(axis='y', linestyle=':', alpha=0.5)

# Add value labels on top of the bars
for p in ax_bar.patches:
    ax_bar.annotate(f"{p.get_height():,.0f} Billion VND", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0, 5), textcoords='offset points')

# Save high-resolution chart
output_projection_path = os.path.join(assets_dir, 'revenue_projection_130_bep.png')
plt.savefig(output_projection_path, dpi=300, bbox_inches='tight')
print(f"Projection chart successfully exported to: '{output_projection_path}'")

plt.show()

#### Key Finding

Setting a 130% BEP volume ceiling on the Website channel would reduce total corporate revenue by **25.61%** (-2.2 T VND) relative to 2024, as the channel is currently operating at **95.8× above** its BEP threshold. BEP is an appropriate risk floor metric; it is not suitable as a forward growth target for a channel already at scale.

## 4. Strategy A+ Financial Simulations (2026 Forecast)

This section models consolidated corporate net profit for 2026 under the proposed Strategy A+ restructuring plan, evaluated under two scenarios relative to the 2024 baseline (Net Profit: 6.101 T VND).

### 4.1. Simulation Parameters

**Negative impacts (structural transition costs):**

| Item | Impact |
|------|--------|
| Phase-out of Product B | -403 B VND |
| Short-term store closure friction | -50 B VND |

**Positive impacts (savings & reinvestment):**

| Lever | Base Case | Optimistic Case |
|-------|-----------|-----------------|
| Store closure (fixed cost savings) | 30% closure → +251 B VND | 35% closure → +293 B VND |
| Commission renegotiation | 15% reduction → +75 B VND | 20% reduction → +93 B VND |
| Website reinvestment (50% of savings, ROAS 5:1, at 86.56% net margin) | +545 B VND | +759 B VND (ROAS 6:1) |

### 4.2. Projected Outcomes

| Scenario | 2026 Net Profit | YoY Change |
|----------|----------------|------------|
| Baseline (2024 actual) | 6,101 B VND | — |
| Strategy A+ — Base Case | 6,519 B VND | +6.8% |
| Strategy A+ — Optimistic Case | 6,793 B VND | +11.3% |

In [ ]:
print("--- 1. Simulating Strategy A+ Scenarios (Base vs. Optimistic) ---")

# Define assets directory path resolution locally
import os
import matplotlib.pyplot as plt
import seaborn as sns

assets_dir = '../assets/charts/'
if not os.path.exists('../assets/'):
    assets_dir = 'assets/charts/'
os.makedirs(assets_dir, exist_ok=True)

# Baseline 2024 Profit
baseline_profit_2024 = 6_101_067_066_645 / 1e9 # Convert to Billion VND

# Shared losses
loss_product_b = -403.0   # -403 Billion VND
loss_closures = -50.0     # -50 Billion VND

# --- A+ Base Case Calculations ---
savings_fc_base = 251.0
savings_comm_base = 75.0
marketing_reinvest_profit_base = 545.0 # Shifting budget with ROAS 5:1

net_impact_base = loss_product_b + loss_closures + savings_fc_base + savings_comm_base + marketing_reinvest_profit_base
projected_profit_base = baseline_profit_2024 + net_impact_base
growth_base = (net_impact_base / baseline_profit_2024) * 100

# --- A+ Optimistic Case Calculations ---
savings_fc_opt = 293.0
savings_comm_opt = 93.0
marketing_reinvest_profit_opt = 759.0 # Shifting budget with ROAS 6:1

net_impact_opt = loss_product_b + loss_closures + savings_fc_opt + savings_comm_opt + marketing_reinvest_profit_opt
projected_profit_opt = baseline_profit_2024 + net_impact_opt
growth_opt = (net_impact_opt / baseline_profit_2024) * 100

print(f"Base Case Net Impact: +{net_impact_base:.1f} Billion VND | Growth: +{growth_base:.2f}%")
print(f"Optimistic Case Net Impact: +{net_impact_opt:.1f} Billion VND | Growth: +{growth_opt:.2f}%")


print("\n--- 2. Consolidating Strategic Scenarios ---")

df_strategy = pd.DataFrame({
    'Scenario': [
        '2024 Baseline\n(Actual)',
        'Strategy A+\n(Base Case)',
        'Strategy A+\n(Optimistic Case)'
    ],
    'Net Profit (Billion VND)': [
        baseline_profit_2024,
        projected_profit_base,
        projected_profit_opt
    ],
    'YoY Growth (%)': [
        0.0,
        growth_base,
        growth_opt
    ]
})

display(df_strategy.style.format({
    'Net Profit (Billion VND)': '{:,.1f} Billion VND',
    'YoY Growth (%)': '{:+.2f}%'
}))


print("\n--- 3. Visualizing Strategic Profit Growth ---")

plt.figure(figsize=(12, 8))
ax_bar = sns.barplot(
    data=df_strategy,
    x='Scenario',
    y='Net Profit (Billion VND)',
    hue='Scenario',
    legend=False,
    palette=['#7F8C8D', '#5CB85C', '#2C3E50'] # Muted Grey, Growth Green, Deep Navy
)

# Customize plot
plt.title('FORECAST OF TOTAL COMPANY NET PROFIT FOR 2026 UNDER STRATEGY A+', fontsize=14, fontweight='bold', pad=25)
plt.ylabel('Net Profit (Billion VND)', fontsize=12)
plt.xlabel('')
plt.ylim(0, max(df_strategy['Net Profit (Billion VND)']) * 1.2)
plt.grid(axis='y', linestyle=':', alpha=0.5)

# Add value labels and YoY growth on top of the bars
for i, p in enumerate(ax_bar.patches):
    height = p.get_height()
    growth_label = f" ({df_strategy['YoY Growth (%)'][i]:+.1f}%)" if i > 0 else ""
    ax_bar.annotate(f"{height:,.1f} Billion VND{growth_label}", 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0, 5), textcoords='offset points')

# Save high-resolution chart
output_strategy_path = os.path.join(assets_dir, 'strategy_aplus_comparison.png')
plt.savefig(output_strategy_path, dpi=300, bbox_inches='tight')
print(f"Strategy comparison chart successfully exported to: '{output_strategy_path}'")

plt.show()

#### Key Findings

Under Strategy A+, net profit improvement ranges from **+418 B VND (+6.8%)** in the base case to **+692 B VND (+11.3%)** in the optimistic case relative to the 2024 baseline. The dominant value driver in both scenarios is the Website channel reinvestment, which leverages a high contribution margin (~94%) and favorable ROAS to amplify savings into disproportionate profit gains.

---

## 5. Conclusions & Strategic Implications

The forward-looking simulations across Modules 2–4 yield three actionable findings:

**Finding 1: Website BEP is structurally low and already exceeded**
The 2025 BEP of 1,944 orders represents less than 1.1% of 2024 actual Website volume. The channel is deep in the profitable zone; BEP analysis is relevant only as a risk floor, not a planning target.

**Finding 2: BEP-as-target is a category error with severe downside**
Constraining Website volume to 130% of BEP would produce a -25.61% corporate revenue contraction. Growth planning for a scaled channel must use absolute volume and margin targets, not BEP multiples.

**Finding 3: Capital reallocation from physical to digital is the highest-return lever**
Strategy A+ base case delivers +6.8% net profit growth by closing 30% of stores and reinvesting savings into the Website channel. The optimistic case (+11.3%) is achievable by increasing the reallocation rate and negotiating higher commission reductions. In both cases, the Website reinvestment component accounts for the majority of incremental profit, confirming the channel shift hypothesis established in Notebook 2, Section 7.